# Markov Random Fields

**Companion lesson:** https://ml-viz.vercel.app/courses/graphical-models/02-markov-random-fields

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A 2-node Ising MRF — potentials and the partition function

Two spins that prefer to align: $\psi(x_1,x_2)=e^{\beta x_1 x_2}$. We compute the distribution explicitly, including the partition function $Z$.

In [ ]:
beta = 1.0
states = [(+1,+1), (-1,-1), (+1,-1), (-1,+1)]
psi = np.array([np.exp(beta*a*b) for a,b in states])
Z = psi.sum()
for (a,b), p in zip(states, psi/Z):
    print(f'x1={a:+d} x2={b:+d}: P = {p:.3f}')
print('P(aligned) =', round((psi[0]+psi[1])/Z, 3), ' (Z =', round(Z,3), ')')

## Image denoising — an MRF you can see

Each pixel is a spin. Energy = a **data term** (stay close to the noisy observation) + a **smoothness term** (agree with neighbors). We minimize it with Iterated Conditional Modes (ICM): repeatedly set each pixel to its lowest-energy value given its neighbors.

In [ ]:
rng = np.random.RandomState(0)
# clean binary image: a filled square
img = -np.ones((40, 40)); img[10:30, 10:30] = 1
noisy = img.copy()
flip = rng.rand(*img.shape) < 0.15            # 15% salt-and-pepper noise
noisy[flip] *= -1

def denoise(y, eta=2.0, beta=2.0, sweeps=6):
    x = y.copy()
    H, W = x.shape
    for _ in range(sweeps):
        for i in range(H):
            for j in range(W):
                nb = 0
                if i>0: nb += x[i-1,j]
                if i<H-1: nb += x[i+1,j]
                if j>0: nb += x[i,j-1]
                if j<W-1: nb += x[i,j+1]
                # energy(x_ij=+1) vs (-1); pick lower energy
                e_pos = -eta*y[i,j]*1 - beta*nb*1
                e_neg = -eta*y[i,j]*-1 - beta*nb*-1
                x[i,j] = 1 if e_pos < e_neg else -1
    return x

clean = denoise(noisy)
err_before = (noisy != img).mean(); err_after = (clean != img).mean()
print(f'pixel error: noisy {err_before:.3f} -> denoised {err_after:.3f}')
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [img, noisy, clean], ['original','noisy (15%)','MRF denoised']):
    a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()

## Key takeaways

- MRFs score configurations with **potentials** over cliques; $Z$ normalizes them.
- Writing potentials as $e^{-\text{energy}}$ connects MRFs to physics (the Ising model).
- Image denoising = minimize data + smoothness energy; ICM does it pixel-by-pixel.
- The smoothness prior cleans noise while preserving the underlying shape.